In [1]:
import torch
import whisper
from transformers import CLIPTokenizer, CLIPTextModel

/home/asdf/anaconda3/envs/datasci/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:295: UserWarning: Failed to initialize NumPy: module 'numpy._globals' has no attribute '_signature_descriptor' (Triggered internally at /opt/conda/conda-bld/pytorch_1729647329220/work/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


ImportError: cannot load module more than once per process

ImportError: numpy._core.multiarray failed to import

In [ ]:

class TextDeceptionPipeline:
    def __init__(self, whisper_size="base", clip_model="openai/clip-vit-base-patch32"):
        """
        Initializes both the ASR (Whisper) and Embedding (CLIP) models.
        """
        print(f"Loading Whisper: {whisper_size}...")
        self.asr_model = whisper.load_model(whisper_size)
        
        print(f"Loading CLIP: {clip_model}...")
        # We use the HuggingFace implementation to easily access 'last_hidden_state'
        # separate from the pooling layer.
        self.tokenizer = CLIPTokenizer.from_pretrained(clip_model)
        self.text_encoder = CLIPTextModel.from_pretrained(clip_model)
        self.text_encoder.eval() # Freeze layers

    def process_audio(self, audio_path):
        """
        Full pipeline: Audio -> Timestamped Words -> CLIP Embeddings
        """
        
        # --- Step 1: Transcribe with Timestamps ---
        # The 'word_timestamps=True' flag is the critical hinge here.
        result = self.asr_model.transcribe(audio_path, word_timestamps=True)
        
        full_text_data = []
        
        # Whisper breaks audio into 'segments' (sentences/phrases)
        for segment in result["segments"]:
            # Each segment contains a list of 'words'
            for word_obj in segment["words"]:
                word_text = word_obj["word"].strip()
                start_time = word_obj["start"]
                end_time = word_obj["end"]
                
                # Filter out empty noise or silence tokens
                if not word_text:
                    continue
                    
                full_text_data.append({
                    "word": word_text,
                    "start": start_time,
                    "end": end_time
                })
                
        return self._embed_and_align(full_text_data)

    def _embed_and_align(self, word_data):
        """
        Converts a list of {word, start, end} into a Tensor of aligned embeddings.
        """
        
        # 1. Prepare raw text batch
        words = [w["word"] for w in word_data]
        timestamps = [(w["start"], w["end"]) for w in word_data]
        
        # 2. Tokenize
        # CLIP Tokenizer will split "understanding" -> "under", "standing"
        # We need to preserve context, so we tokenize the whole sequence, 
        # but we must track which embedding belongs to which timestamp.
        
        # Join words to form a sentence context (CLIP likes sentences, not isolated words)
        text_context = " ".join(words)
        
        inputs = self.tokenizer(
            text_context, 
            return_tensors="pt", 
            padding=True, 
            truncation=True,
            max_length=77 # CLIP's hard limit
        )
        
        # 3. Generate Embeddings (The 'Query' Vectors)
        with torch.no_grad():
            outputs = self.text_encoder(**inputs)
        
        # outputs.last_hidden_state shape: (Batch, Seq_Len, 512)
        # We perform a "Squeeze" to remove batch dim since we process one sequence here
        token_embeddings = outputs.last_hidden_state.squeeze(0) 
        
        # 4. Re-Align Timestamps (The tricky part)
        # The tokenizer output includes [CLS] and [SEP] tokens which we don't want.
        # We also need to map the original word timestamps to the sub-word tokens.
        
        aligned_embeddings = []
        aligned_timestamps = []
        
        # Get mapping of token -> original word index
        # word_ids() returns [None, 0, 1, 1, 2, None] for "[CLS] A B B C [SEP]"
        word_indices = inputs.word_ids() 
        
        for idx, word_idx in enumerate(word_indices):
            if word_idx is None:
                # Skip [CLS] and [SEP] tokens for the alignment map
                continue
            
            # Get the embedding for this token
            emb = token_embeddings[idx] # Shape: (512,)
            
            # Get the timestamp of the original word this token belongs to
            ts = timestamps[word_idx]
            
            aligned_embeddings.append(emb)
            aligned_timestamps.append(ts)
            
        # Stack into final tensors
        final_embeddings = torch.stack(aligned_embeddings) # (N_tokens, 512)
        
        # Returns:
        # 1. Tensor of CLIP embeddings (The Queries)
        # 2. List of (Start, End) tuples for temporal attention masking
        return final_embeddings, aligned_timestamps

# --- Example Usage ---
if __name__ == "__main__":
    # Note: Requires an actual audio file to run completely
    # pipeline = TextDeceptionPipeline(whisper_size="tiny")
    # embs, times = pipeline.process_audio("interrogation_sample.mp3")
    
    # print(f"Generated {len(embs)} embeddings.")
    # print(f"First word timestamp: {times[0]}")
    pass